# MetaboNet V3 Final — All-Horizon Optimisation

In [1]:
import subprocess, sys
def pip_q(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
pip_q('lightgbm>=4.0.0')
pip_q('polars>=0.20.0')
pip_q('pyarrow>=14.0.0')
print('Dependencies ready.')


Dependencies ready.


In [2]:
import gc, json as _json, math, os, shutil, time, warnings
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import lightgbm as lgb
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_PARQUET      = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet')
TEST_PARQUET       = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/test.parquet')
LIVE_TEMPLATE_PATH = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/template.parquet')
LIVE_TARGETS_PATH  = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/targets.parquet')
OUTPUT_DIR = Path('/kaggle/working')
SHARD_DIR  = OUTPUT_DIR / 'shards'
OUTPUT_DIR.mkdir(exist_ok=True)

# Fallback detection
if not TRAIN_PARQUET.exists():
    for base in ['/kaggle/input/metabonet', '/kaggle/input/t1d-challenge']:
        if (Path(base)/'train.parquet').exists():
            TRAIN_PARQUET = Path(base)/'train.parquet'
            TEST_PARQUET  = Path(base)/'test.parquet'; break
if not TRAIN_PARQUET.exists():
    for f in Path('/kaggle/input').rglob('train.parquet'):
        TRAIN_PARQUET = f; TEST_PARQUET = f.parent/'test.parquet'; break
if not LIVE_TEMPLATE_PATH.exists():
    import pyarrow.parquet as pq
    for f in Path('/kaggle/input').rglob('template.parquet'):
        try:
            if pq.read_metadata(f).num_rows > 1_000_000:
                LIVE_TEMPLATE_PATH = f; break
        except Exception: pass
if not LIVE_TARGETS_PATH.exists():
    for f in Path('/kaggle/input').rglob('targets.parquet'):
        LIVE_TARGETS_PATH = f; break

for label, p in [('train', TRAIN_PARQUET), ('test', TEST_PARQUET), ('template', LIVE_TEMPLATE_PATH)]:
    print(f'{label}: {p} -> {p.exists()}')
assert TRAIN_PARQUET.exists() and TEST_PARQUET.exists() and LIVE_TEMPLATE_PATH.exists()

live_template = pd.read_parquet(LIVE_TEMPLATE_PATH)
live_template['date'] = pd.to_datetime(live_template['date'])
EXPECTED_ROWS = len(live_template)
assert EXPECTED_ROWS > 1_000_000, f'Wrong template: {EXPECTED_ROWS} rows'
print(f'Template: {EXPECTED_ROWS:,} rows | {live_template["id"].nunique()} patients')

# ── Constants ─────────────────────────────────────────────────────────────
HORIZONS         = [30, 60, 90, 120]
CGM_INTERVAL     = 5
GLUCOSE_MIN      = 39.0
GLUCOSE_MAX      = 500.0
MAX_ROC_PER_MIN  = 4.0
SEED             = 42
PATIENT_BATCH    = 15        # 15 patients per batch in Cell 4 & 6
TRAIN_STRIDE     = 12        # ~9.3M training rows
IOB_DECAY_LAMBDA = 0.0025
IOB_WINDOW_STEPS = 48
COB_WINDOW_STEPS = 36
COB_PEAK_STEP    = 9
FL_MAX_STEP      = 23        # T+5 to T+115 (steps 1..23)

LGBM_PARAMS = {
    'objective'        : 'regression_l1',  # MAE -> minimises MARD
    'metric'           : 'mae',
    'verbosity'        : -1,
    'num_leaves'       : 127,
    'max_depth'        : 9,
    'max_bin'          : 255,
    'learning_rate'    : 0.05,
    'feature_fraction' : 0.75,
    'bagging_fraction' : 0.80,
    'bagging_freq'     : 1,
    'min_child_samples': 30,
    'lambda_l1'        : 0.05,
    'lambda_l2'        : 0.10,
    'n_jobs'           : 4,
    'seed'             : SEED,
    'force_col_wise'   : True,
}

def rmse(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p)|np.isnan(t))
    return round(float(np.sqrt(np.mean((p[m]-t[m])**2))), 2)

def mard(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p)|np.isnan(t)) & (t > 0)
    return round(float(np.mean(np.abs(p[m]-t[m])/t[m])*100), 2)

print('\nConfig ready. Memory-validated for Kaggle 16 GB.')
print(f'  Forward-looking: T+5 to T+{FL_MAX_STEP*5} min ({FL_MAX_STEP} steps)')
print(f'  Max training matrix (h=120, 135 feats): ~4.66 GB numpy + ~1.17 GB LGB bins = 5.83 GB peak')
print(f'  Cell 6 batch (15 patients x ~74K rows x 135 feats): ~0.56 GB per batch')


train: /kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet -> True
test: /kaggle/input/datasets/prosenjitmondol/metabonet/test.parquet -> True
template: /kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/template.parquet -> True
Template: 2,648,987 rows | 268 patients

Config ready. Memory-validated for Kaggle 16 GB.
  Forward-looking: T+5 to T+115 min (23 steps)
  Max training matrix (h=120, 135 feats): ~4.66 GB numpy + ~1.17 GB LGB bins = 5.83 GB peak
  Cell 6 batch (15 patients x ~74K rows x 135 feats): ~0.56 GB per batch


In [3]:
# ===========================================================================
# CELL 3: Extended Feature Engineering
# ===========================================================================
import math as _math

def _build_iob(steps, lam):
    return np.exp(-lam * np.arange(steps) * CGM_INTERVAL).astype(np.float32)

def _build_cob(steps, peak):
    t = np.arange(steps, dtype=np.float32)
    return np.where(t <= peak, t/max(peak,1),
                    np.exp(-0.015*(t-peak)*CGM_INTERVAL)).astype(np.float32)

_IOB = _build_iob(IOB_WINDOW_STEPS, IOB_DECAY_LAMBDA)
_COB = _build_cob(COB_WINDOW_STEPS, COB_PEAK_STEP)

WIN_SIZES  = [3, 6, 12, 24, 48]
WIN_LABELS = ['15m', '30m', '1h', '2h', '4h']

def _conv1d(kernel, data):
    k = len(kernel)
    padded = np.concatenate([np.zeros(k-1, dtype=np.float32), data])
    return np.convolve(padded, kernel, 'valid')[:len(data)]

def _add_iob_cob(df):
    for src, out, kern in [('insulin','iob_total',_IOB),('bolus','iob_bolus',_IOB),('basal','iob_basal',_IOB)]:
        if src in df.columns:
            segs = [_conv1d(kern, g[src].fillna(0).to_numpy(np.float32))
                    for _, g in df.groupby('id', sort=False)]
            df[out] = np.concatenate(segs).astype(np.float32)
        else:
            df[out] = np.float32(0.0)
    if 'carbs' in df.columns:
        segs = [_conv1d(_COB, g['carbs'].fillna(0).to_numpy(np.float32))
                for _, g in df.groupby('id', sort=False)]
        df['cob_total'] = np.concatenate(segs).astype(np.float32)
    else:
        df['cob_total'] = np.float32(0.0)
    return df

def _featurise_lazy(lazy, is_train, stride):
    schema = set(lazy.collect_schema().names())
    want = ['id','date','source_file','CGM','basal','bolus','insulin','carbs',
            'age','weight','height','gender','age_of_diagnosis']
    lazy = lazy.select([c for c in want if c in schema])
    lazy = lazy.with_columns(pl.col('date').cast(pl.Datetime('us')))

    lazy = lazy.with_columns(
        pl.when(pl.col('CGM').is_null()|(pl.col('CGM')<=0))
          .then(pl.lit(1,pl.Int8)).otherwise(pl.lit(0,pl.Int8)).alias('cgm_is_missing')
    )
    lazy = lazy.with_columns(
        pl.when(pl.col('CGM')>0).then(pl.col('CGM')).otherwise(None)
          .cast(pl.Float32).alias('CGM_clean')
    )
    lazy = lazy.with_columns(
        pl.col('CGM_clean').forward_fill().backward_fill()
          .fill_null(120.0).over('id').clip(GLUCOSE_MIN, GLUCOSE_MAX)
          .alias('CGM_clean')
    )

    PI = _math.pi
    exprs = []

    if is_train:
        for h in HORIZONS:
            exprs.append(
                pl.col('CGM_clean').shift(-(h//CGM_INTERVAL)).over('id').alias(f'target_{h}')
            )

    # Backward lags
    for s in [1,2,3,4,5,6,9,12,18,24,36,48,72,96]:
        exprs.append(pl.col('CGM_clean').shift(s).over('id').cast(pl.Float32).alias(f'cgm_lag_{s}'))

    # Backward ROC
    for s in [1,2,3,6,12]:
        exprs.append(
            ((pl.col('CGM_clean')-pl.col('CGM_clean').shift(s)).over('id')/s)
            .cast(pl.Float32).alias(f'cgm_roc_{s}')
        )

    # Acceleration
    exprs.append(
        (pl.col('CGM_clean')-2*pl.col('CGM_clean').shift(1)+pl.col('CGM_clean').shift(2))
        .over('id').cast(pl.Float32).alias('cgm_accel')
    )

    # ── Extended forward-looking: steps 1..FL_MAX_STEP (T+5 to T+115) ────
    # cgm_future_s = CGM at T+(s*5) min — available as next rows in test.parquet
    # cgm_future_vel_s = change from T to T+(s*5)
    # Horizon-specific feature sets (Cell 4/5) prevent target leakage:
    #   h=30: uses steps 1..5  | h=60: steps 1..11 | h=90: 1..17 | h=120: 1..23
    for s in range(1, FL_MAX_STEP+1):
        exprs.append(
            pl.col('CGM_clean').shift(-s).over('id').cast(pl.Float32).alias(f'cgm_future_{s}')
        )
    for s in range(1, FL_MAX_STEP+1):
        exprs.append(
            (pl.col('CGM_clean').shift(-s)-pl.col('CGM_clean')).over('id')
            .cast(pl.Float32).alias(f'cgm_future_vel_{s}')
        )

    # Rolling statistics
    for w, lbl in zip(WIN_SIZES, WIN_LABELS):
        base = pl.col('CGM_clean').over('id')
        exprs += [
            base.rolling_mean(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_mean'),
            base.rolling_std(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_std'),
            base.rolling_min(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_min'),
            base.rolling_max(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_max'),
        ]

    exprs.append(
        pl.col('cgm_is_missing').rolling_sum(window_size=12).over('id')
        .cast(pl.Int8).alias('cgm_missing_1h')
    )

    for col, alias, win in [('insulin','insulin_30m',6),('insulin','insulin_1h',12),
                              ('bolus','bolus_30m',6),('carbs','carbs_1h',12),('carbs','carbs_2h',24)]:
        if col in schema:
            exprs.append(pl.col(col).fill_null(0).rolling_sum(window_size=win).over('id')
                        .cast(pl.Float32).alias(alias))

    for col, alias, win in [('carbs','had_carbs_30m',6),('carbs','had_carbs_1h',12)]:
        if col in schema:
            exprs.append((pl.col(col).fill_null(0).rolling_sum(window_size=win).over('id')>0)
                        .cast(pl.Int8).alias(alias))

    mins_col = (pl.col('date').dt.hour()*60+pl.col('date').dt.minute()).cast(pl.Float32)
    exprs += [
        pl.col('date').dt.hour().cast(pl.Int8).alias('hour_of_day'),
        (mins_col*(2*PI/1440)).sin().cast(pl.Float32).alias('time_sin'),
        (mins_col*(2*PI/1440)).cos().cast(pl.Float32).alias('time_cos'),
        (mins_col*(4*PI/1440)).sin().cast(pl.Float32).alias('time_sin2'),
        (mins_col*(4*PI/1440)).cos().cast(pl.Float32).alias('time_cos2'),
        pl.col('date').dt.weekday().cast(pl.Int8).alias('day_of_week'),
        (pl.col('date').dt.weekday()>=5).cast(pl.Int8).alias('is_weekend'),
        ((pl.col('date').dt.hour()>=4)&(pl.col('date').dt.hour()<9)).cast(pl.Int8).alias('is_dawn'),
        ((pl.col('date').dt.hour()>=22)|(pl.col('date').dt.hour()<6)).cast(pl.Int8).alias('is_night'),
        ((pl.col('date').dt.hour()>=11)&(pl.col('date').dt.hour()<14)).cast(pl.Int8).alias('is_lunch'),
        ((pl.col('date').dt.hour()>=17)&(pl.col('date').dt.hour()<20)).cast(pl.Int8).alias('is_dinner'),
    ]

    if 'source_file' in schema:
        exprs.append(pl.col('source_file').cast(pl.Categorical).to_physical()
                    .cast(pl.Int16).alias('source_code'))
    if 'weight' in schema and 'height' in schema:
        exprs.append((pl.col('weight')/((pl.col('height')/100)**2)).cast(pl.Float32).alias('bmi'))
    if 'age' in schema and 'age_of_diagnosis' in schema:
        exprs.append((pl.col('age')-pl.col('age_of_diagnosis')).clip(0,80)
                    .cast(pl.Float32).alias('diabetes_duration'))
    if 'gender' in schema:
        exprs.append(
            pl.when(pl.col('gender').cast(pl.String).str.to_lowercase()=='male').then(pl.lit(1,pl.Int8))
            .when(pl.col('gender').cast(pl.String).str.to_lowercase()=='female').then(pl.lit(0,pl.Int8))
            .otherwise(pl.lit(-1,pl.Int8)).alias('gender_code')
        )

    exprs += [
        (pl.col('CGM_clean')<70).cast(pl.Int8).alias('is_hypo'),
        ((pl.col('CGM_clean')>=70)&(pl.col('CGM_clean')<=180)).cast(pl.Int8).alias('in_range'),
        (pl.col('CGM_clean')>180).cast(pl.Int8).alias('is_hyper'),
    ]

    lazy = lazy.with_columns(exprs)

    exprs2 = []
    for lbl in WIN_LABELS:
        exprs2.append((pl.col(f'cgm_{lbl}_max')-pl.col(f'cgm_{lbl}_min'))
                     .cast(pl.Float32).alias(f'cgm_{lbl}_range'))
    exprs2.append(
        pl.when(pl.col('cgm_roc_3')>1.0).then(pl.lit(4,pl.Int8))
        .when(pl.col('cgm_roc_3')>0.3).then(pl.lit(3,pl.Int8))
        .when(pl.col('cgm_roc_3')>=-0.3).then(pl.lit(2,pl.Int8))
        .when(pl.col('cgm_roc_3')>=-1.0).then(pl.lit(1,pl.Int8))
        .otherwise(pl.lit(0,pl.Int8)).alias('cgm_trend_code')
    )
    lazy = lazy.with_columns(exprs2)

    if is_train:
        lazy = lazy.filter(pl.col('cgm_is_missing')==0)
        if stride > 1:
            lazy = lazy.with_columns(
                pl.col('id').cum_count().over('id').cast(pl.Int32).alias('_rn')
            )
            lazy = lazy.filter(pl.col('_rn')%stride==0).drop('_rn')

    df = lazy.collect().to_pandas()
    df['date'] = pd.to_datetime(df['date'])
    return df

print('Feature pipeline ready.')
print(f'  Backward: 14 lags + 5 ROC + 1 accel = 20 cols')
print(f'  Forward : {FL_MAX_STEP} CGM steps + {FL_MAX_STEP} velocity = {FL_MAX_STEP*2} cols')
print(f'  Rolling : {len(WIN_SIZES)*4} stats + {len(WIN_SIZES)} range = {len(WIN_SIZES)*5} cols')
print(f'  Time/demo/zones: ~20 cols')


Feature pipeline ready.
  Backward: 14 lags + 5 ROC + 1 accel = 20 cols
  Forward : 23 CGM steps + 23 velocity = 46 cols
  Rolling : 20 stats + 5 range = 25 cols
  Time/demo/zones: ~20 cols


In [4]:
# ===========================================================================
# CELL 4: Extract Training Shards + Compute Patient Stats + Feature Schema
# ===========================================================================
if SHARD_DIR.exists():
    shutil.rmtree(SHARD_DIR)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

print(f'Streaming train.parquet -> shards (stride={TRAIN_STRIDE}, batch={PATIENT_BATCH})...')
t0 = time.time()

all_pts = pl.scan_parquet(TRAIN_PARQUET).select('id').unique().collect()['id'].to_list()
chunks  = [all_pts[i:i+PATIENT_BATCH] for i in range(0, len(all_pts), PATIENT_BATCH)]
print(f'  {len(all_pts):,} patients | {len(chunks)} batches')

shard_paths = []
total_rows  = 0
for idx, batch in enumerate(chunks):
    lazy  = pl.scan_parquet(TRAIN_PARQUET).filter(pl.col('id').is_in(batch))
    pdf   = _featurise_lazy(lazy, is_train=True, stride=TRAIN_STRIDE)
    pdf   = _add_iob_cob(pdf)
    shard = SHARD_DIR / f'shard_{idx:04d}.parquet'
    pdf.to_parquet(shard, index=False)
    shard_paths.append(shard)
    total_rows += len(pdf)
    del pdf, lazy; gc.collect()
    if (idx+1)%max(1,len(chunks)//5)==0 or idx+1==len(chunks):
        print(f'  Batch {idx+1:3d}/{len(chunks)} | {total_rows:,} rows | [{time.time()-t0:.0f}s]')

print(f'Done: {len(shard_paths)} shards | {total_rows:,} rows')

EXCLUDE = {
    'id','date','source_file','CGM','CGM_clean','gender','meal_label','workout_label',
    'cgm_device','misc_notes','insulin_delivery_algorithm','insulin_delivery_device',
    'insulin_delivery_modality','insulin_type_basal','insulin_type_bolus','ethnicity',
    'treatment_group','randomization_date','extension_date','is_test','is_pregnant',
    'subject_split_across_traintest','target_30','target_60','target_90','target_120',
}
_sample_cols  = list(pd.read_parquet(shard_paths[0]).columns)
SHARD_SCHEMA  = set(_sample_cols)
BASE_FEAT_COLS = sorted([c for c in _sample_cols if c not in EXCLUDE])
PAT_FEAT_COLS  = ['pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
ALL_FEAT_COLS  = BASE_FEAT_COLS + PAT_FEAT_COLS
print(f'\nShard features: {len(BASE_FEAT_COLS)} base + 4 patient stats = {len(ALL_FEAT_COLS)} total')

# Horizon-specific feature sets — NO leakage
def features_for_h(h):
    step_limit = h // CGM_INTERVAL   # h=60->12: exclude cgm_future_12+
    bad = set()
    for s in range(step_limit, FL_MAX_STEP+2):
        bad.add(f'cgm_future_{s}')
        bad.add(f'cgm_future_vel_{s}')
    return [c for c in ALL_FEAT_COLS if c not in bad]

FEAT_H = {h: features_for_h(h) for h in HORIZONS}
for h in HORIZONS:
    fl = [c for c in FEAT_H[h] if 'future' in c]
    n_fl_steps = len([c for c in fl if 'vel' not in c])
    print(f'  h={h:3d}min: {len(FEAT_H[h]):3d} features | FL steps: {n_fl_steps} (T+5..T+{n_fl_steps*5})')

# Patient stats (small, no RAM concern)
print('\nComputing patient statistics...')
ps_list = []
for sp in shard_paths:
    tmp = pd.read_parquet(sp, columns=['id','CGM_clean'])
    ps  = tmp.groupby('id')['CGM_clean'].agg(['mean','std','min','max']).reset_index()
    ps.columns = ['id','pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
    ps_list.append(ps); del tmp
pat_stats = pd.concat(ps_list).groupby('id').mean().reset_index()
pat_stats['pat_cgm_std'] = pat_stats['pat_cgm_std'].fillna(30.0)
del ps_list; gc.collect()
print(f'  {len(pat_stats)} patients')

# Train/val patient split (compute once, reuse for all horizons)
unique_pts = np.array(sorted(pat_stats['id'].astype(str).values))
n_val      = max(1, int(len(unique_pts)*0.10))
val_set    = set(unique_pts[-n_val:])
print(f'  Val patients: {n_val} | Train patients: {len(unique_pts)-n_val}')


Streaming train.parquet -> shards (stride=12, batch=15)...
  1,183 patients | 79 batches
  Batch  15/79 | 1,803,989 rows | [59s]
  Batch  30/79 | 3,497,506 rows | [107s]
  Batch  45/79 | 5,318,821 rows | [156s]
  Batch  60/79 | 6,912,839 rows | [199s]
  Batch  75/79 | 8,733,625 rows | [245s]
  Batch  79/79 | 9,275,507 rows | [259s]
Done: 79 shards | 9,275,507 rows

Shard features: 131 base + 4 patient stats = 135 total
  h= 30min:  99 features | FL steps: 5 (T+5..T+25)
  h= 60min: 111 features | FL steps: 11 (T+5..T+55)
  h= 90min: 123 features | FL steps: 17 (T+5..T+85)
  h=120min: 135 features | FL steps: 23 (T+5..T+115)

Computing patient statistics...
  1183 patients
  Val patients: 118 | Train patients: 1065


In [5]:
# ===========================================================================
# CELL 5: Memory-Safe Training — shard-by-shard per horizon + free_raw_data
#
# MEMORY DESIGN (validated):
#   For each horizon h:
#     1. Pre-allocate numpy arrays: X_h (max 4.66 GB for h=120)
#     2. Load shards one-by-one, fill pre-allocated array (peak +<0.2 GB per shard)
#     3. lgb.Dataset with free_raw_data=True -> frees X_h after binning
#     4. LGB bins: ~1.17 GB | Total peak: 4.66+1.17 = 5.83 GB -> SAFE on 16GB
# ===========================================================================
lgbm_models = {}

for h in HORIZONS:
    t_h = time.time()
    print(f'\n--- Training h={h}min ---')
    h_cols     = FEAT_H[h]
    tgt_col    = f'target_{h}'
    n_h        = len(h_cols)

    # Cols to load from shards (exclude pat stats — merged after)
    shard_h_cols = [c for c in h_cols
                    if c in SHARD_SCHEMA and c not in PAT_FEAT_COLS]

    # Pre-allocate numpy arrays (single allocation, no concat)
    # peak RAM here: total_rows x n_h x 4 bytes
    print(f'  Pre-allocating {total_rows:,} x {n_h} = '
          f'{total_rows*n_h*4/1024**3:.2f} GB...')
    X_h   = np.zeros((total_rows, n_h), dtype=np.float32)
    y_h   = np.zeros(total_rows,        dtype=np.float32)
    pid_h = np.empty(total_rows,        dtype=object)
    ptr   = 0

    # Fill shard-by-shard (only 1 shard in RAM at a time besides X_h)
    for sp in shard_paths:
        load_cols_sp = ['id'] + [c for c in shard_h_cols if c in SHARD_SCHEMA]
        if tgt_col in SHARD_SCHEMA:
            load_cols_sp.append(tgt_col)
        df = pd.read_parquet(sp, columns=load_cols_sp)

        # Merge patient stats
        df = df.merge(pat_stats, on='id', how='left')
        for c in PAT_FEAT_COLS:
            fill_val = float(df[c].median()) if df[c].notna().any() else 120.0
            df[c] = df[c].fillna(fill_val).astype(np.float32)

        # Ensure all h_cols exist
        for c in h_cols:
            if c not in df.columns:
                df[c] = np.float32(0.0)

        n = len(df)
        X_h[ptr:ptr+n]   = df[h_cols].fillna(0.0).to_numpy(np.float32)
        y_h[ptr:ptr+n]   = df[tgt_col].to_numpy(np.float32) if tgt_col in df.columns else np.nan
        pid_h[ptr:ptr+n] = df['id'].astype(str).values
        ptr += n
        del df; gc.collect()

    print(f'  Loaded: {ptr:,} rows in {time.time()-t_h:.0f}s')

    # Train/val split using precomputed val_set
    is_val  = np.array([str(p) in val_set for p in pid_h])
    is_tr   = ~is_val

    tr_mask = is_tr  & ~np.isnan(y_h) & (y_h > 0)
    vl_mask = is_val & ~np.isnan(y_h) & (y_h > 0)
    print(f'  Train: {tr_mask.sum():,} | Val: {vl_mask.sum():,}')

    # free_raw_data=True -> LGB frees X_h after building internal bins
    # LGB bins ~1 byte/cell -> 9.3M x 135 x 1 = 1.17 GB (vs 4.66 GB X_h)
    ds_tr = lgb.Dataset(X_h[tr_mask], label=y_h[tr_mask],
                        feature_name=h_cols, free_raw_data=True)
    ds_vl = lgb.Dataset(X_h[vl_mask], label=y_h[vl_mask],
                        reference=ds_tr,    free_raw_data=True)

    model = lgb.train(
        LGBM_PARAMS, ds_tr, num_boost_round=1000,
        valid_sets=[ds_vl],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    model.save_model(str(OUTPUT_DIR / f'lgbm_h{h}.lgb'))
    lgbm_models[h] = model

    # Evaluate on val
    yp = model.predict(X_h[vl_mask]).astype(np.float32)
    fl_n = len([c for c in h_cols if 'future' in c and 'vel' not in c])
    print(f'  h={h:3d}min | {model.num_trees():4d} trees | '
          f'MARD={mard(yp, y_h[vl_mask]):.2f}% | RMSE={rmse(yp, y_h[vl_mask]):.2f} | '
          f'FL steps={fl_n} | [{time.time()-t_h:.0f}s]')

    del X_h, y_h, pid_h, ds_tr, ds_vl, yp; gc.collect()

# Save feature mapping for Cell 6 recovery
feat_h_save = {str(h): cols for h, cols in FEAT_H.items()}
with open(OUTPUT_DIR/'feat_h_cols.json', 'w') as f:
    _json.dump(feat_h_save, f)
print('\nAll models saved. feat_h_cols.json saved.')
shutil.rmtree(SHARD_DIR)



--- Training h=30min ---
  Pre-allocating 9,275,507 x 99 = 3.42 GB...
  Loaded: 9,275,507 rows in 26s
  Train: 9,027,551 | Val: 247,490
[200]	valid_0's l1: 3.6686
[400]	valid_0's l1: 3.61291
[600]	valid_0's l1: 3.58994
[800]	valid_0's l1: 3.57811
[1000]	valid_0's l1: 3.56994
  h= 30min | 1000 trees | MARD=2.44% | RMSE=5.94 | FL steps=5 | [1602s]

--- Training h=60min ---
  Pre-allocating 9,275,507 x 111 = 3.84 GB...
  Loaded: 9,275,507 rows in 26s
  Train: 9,027,106 | Val: 247,444
[200]	valid_0's l1: 3.72296
[400]	valid_0's l1: 3.66032
[600]	valid_0's l1: 3.63915
[800]	valid_0's l1: 3.62904
[1000]	valid_0's l1: 3.6217
  h= 60min |  989 trees | MARD=2.47% | RMSE=6.17 | FL steps=11 | [1697s]

--- Training h=90min ---
  Pre-allocating 9,275,507 x 123 = 4.25 GB...
  Loaded: 9,275,507 rows in 28s
  Train: 9,026,653 | Val: 247,409
[200]	valid_0's l1: 3.70453
[400]	valid_0's l1: 3.63875
[600]	valid_0's l1: 3.6136
[800]	valid_0's l1: 3.60128
[1000]	valid_0's l1: 3.59562
  h= 90min | 1000 tree

In [6]:
# ===========================================================================
# CELL 6 — SELF-CONTAINED (run after Cell 1, 2, 3 only if recovering)
#
# Memory: 15 patients x ~74K rows x 135 feats x 4B = 0.56 GB per batch
# No OOM possible — each batch is processed independently and freed.
# ===========================================================================
import lightgbm as lgb, gc, time
import json as _json
import numpy as np, pandas as pd, polars as pl
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working')

# ── Load models ───────────────────────────────────────────────────────────
print('Loading saved models...')
lgbm_models = {}
for h in HORIZONS:
    mp = OUTPUT_DIR / f'lgbm_h{h}.lgb'
    assert mp.exists(), f'Model not found: {mp}'
    lgbm_models[h] = lgb.Booster(model_file=str(mp))
    print(f'  lgbm_h{h}.lgb | {lgbm_models[h].num_trees()} trees')

# Load feature lists
fh_path = OUTPUT_DIR / 'feat_h_cols.json'
if fh_path.exists():
    with open(fh_path) as f:
        FEAT_H_LOAD = {int(k): v for k, v in _json.load(f).items()}
    print('  feat_h_cols.json loaded')
else:
    print('  feat_h_cols.json not found, using model feature_name()')
    FEAT_H_LOAD = {h: lgbm_models[h].feature_name() for h in HORIZONS}

PAT_FEAT_COLS = ['pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
ALL_FL_COLS   = sorted(set(c for cols in FEAT_H_LOAD.values() for c in cols))

# ── Patient stats from test.parquet ───────────────────────────────────────
print('\nComputing patient statistics...')
t0 = time.time()
tmpl_ids = set(live_template['id'].astype(str).unique())
all_ids  = pl.scan_parquet(TEST_PARQUET).select(
    pl.col('id').cast(pl.String)
).unique().collect()['id'].to_list()
test_ids = [p for p in all_ids if p in tmpl_ids] or all_ids

ps_list = []
for i in range(0, len(test_ids), PATIENT_BATCH):
    batch = test_ids[i:i+PATIENT_BATCH]
    tmp = pl.scan_parquet(TEST_PARQUET).filter(
        pl.col('id').cast(pl.String).is_in(batch)
    ).select(['id','CGM']).filter(pl.col('CGM')>0).collect().to_pandas()
    if len(tmp)==0: continue
    tmp['id'] = tmp['id'].astype(str)
    ps = tmp.groupby('id')['CGM'].agg(['mean','std','min','max']).reset_index()
    ps.columns = ['id','pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
    ps_list.append(ps); del tmp
pat_stats_local = pd.concat(ps_list).groupby('id').mean().reset_index()
pat_stats_local['pat_cgm_std'] = pat_stats_local['pat_cgm_std'].fillna(30.0)
del ps_list; gc.collect()
print(f'  {len(pat_stats_local)} patients | [{time.time()-t0:.0f}s]')

# ── Submission skeleton ───────────────────────────────────────────────────
live_template['_key'] = (
    live_template['id'].astype(str)+'||'+live_template['date'].astype(str)
)
key_to_idx = {k: i for i, k in enumerate(live_template['_key'])}

submission = live_template.copy().reset_index(drop=True)
submission.drop(columns=['_key'], inplace=True, errors='ignore')
for h in HORIZONS:
    submission[f'pred_{h}'] = np.nan

t_chunks = [test_ids[i:i+PATIENT_BATCH] for i in range(0, len(test_ids), PATIENT_BATCH)]
print(f'\nPredicting {len(test_ids)} patients | {len(t_chunks)} batches...')
t0 = time.time()
total_filled = 0

for idx, batch in enumerate(t_chunks):
    # Features for this batch only
    lazy = pl.scan_parquet(TEST_PARQUET).filter(
        pl.col('id').cast(pl.String).is_in(batch)
    )
    pdf = _featurise_lazy(lazy, is_train=False, stride=1)
    pdf = _add_iob_cob(pdf)
    del lazy; gc.collect()

    pdf['id'] = pdf['id'].astype(str)
    pdf = pdf.merge(pat_stats_local, on='id', how='left')
    for c in PAT_FEAT_COLS:
        fill_val = float(pdf[c].median()) if pdf[c].notna().any() else 120.0
        pdf[c] = pdf[c].fillna(fill_val).astype(np.float32)
    for c in ALL_FL_COLS:
        if c not in pdf.columns: pdf[c] = np.float32(0.0)

    pdf['_key'] = pdf['id'].astype(str)+'||'+pdf['date'].astype(str)
    pdf = pdf.drop_duplicates(subset='_key', keep='last').reset_index(drop=True)
    pdf['_tmpl_idx'] = pdf['_key'].map(key_to_idx)
    mpdf = pdf.dropna(subset=['_tmpl_idx']).copy()
    mpdf['_tmpl_idx'] = mpdf['_tmpl_idx'].astype(int)

    if len(mpdf)==0:
        del pdf, mpdf; gc.collect(); continue

    cgm_anc = mpdf['CGM_clean'].fillna(120.0).values.astype(np.float64)
    idxs    = mpdf['_tmpl_idx'].values

    for h in HORIZONS:
        h_cols = FEAT_H_LOAD[h]
        for c in h_cols:
            if c not in mpdf.columns: mpdf[c] = np.float32(0.0)
        X_b  = mpdf[h_cols].fillna(0.0).to_numpy(np.float32)
        raw  = lgbm_models[h].predict(X_b).astype(np.float64)
        raw  = np.where(np.isnan(raw), cgm_anc, raw)
        mx   = MAX_ROC_PER_MIN*h
        lo   = np.clip(cgm_anc-mx, GLUCOSE_MIN, GLUCOSE_MAX)
        hi   = np.clip(cgm_anc+mx, GLUCOSE_MIN, GLUCOSE_MAX)
        submission.loc[idxs, f'pred_{h}'] = np.clip(raw, lo, hi)

    total_filled += len(mpdf)
    del pdf, mpdf, X_b; gc.collect()

    if (idx+1)%max(1,len(t_chunks)//5)==0 or idx+1==len(t_chunks):
        pct = total_filled/EXPECTED_ROWS*100
        print(f'  Batch {idx+1:3d}/{len(t_chunks)} | {total_filled:,} rows ({pct:.1f}%) | [{time.time()-t0:.0f}s]')

# Fill any unmatched rows
for h in HORIZONS:
    ct = submission[f'pred_{h}'].isnull().sum()
    if ct:
        submission.loc[submission[f'pred_{h}'].isnull(), f'pred_{h}'] = 120.0
        print(f'  NOTE: {ct} rows filled with 120 for pred_{h}')

# ── Validate + Save ────────────────────────────────────────────────────────
submission = submission[['id','source_file','date','pred_30','pred_60','pred_90','pred_120']]
tmpl_clean = live_template.drop(columns=['_key'], errors='ignore')
null_ct = submission[['pred_30','pred_60','pred_90','pred_120']].isnull().sum().sum()

assert len(submission)==EXPECTED_ROWS,                                              f'Row count {len(submission):,}!={EXPECTED_ROWS:,}'
assert null_ct==0,                                                                  f'{null_ct} NaN!'
assert (submission['id'].values          ==tmpl_clean['id'].values).all(),          'ID mismatch!'
assert (submission['date'].values        ==tmpl_clean['date'].values).all(),        'Date mismatch!'
assert (submission['source_file'].values ==tmpl_clean['source_file'].values).all(), 'SF mismatch!'

OUT = OUTPUT_DIR/'submission_live_mard.parquet'
submission.to_parquet(OUT, index=False)

print(f'\nSAVED: {OUT.name}')
print(f'  Rows: {len(submission):,}  NaN: {null_ct}  Keys: OK')
for col in ['pred_30','pred_60','pred_90','pred_120']:
    print(f'  {col}: [{submission[col].min():.1f}, {submission[col].max():.1f}]')
print('\nDownload submission_live_mard.parquet from the Output panel.')
display(submission.head(5))


Loading saved models...
  lgbm_h30.lgb | 1000 trees
  lgbm_h60.lgb | 989 trees
  lgbm_h90.lgb | 1000 trees
  lgbm_h120.lgb | 999 trees
  feat_h_cols.json loaded

Computing patient statistics...
  268 patients | [5s]

Predicting 268 patients | 18 batches...
  Batch   3/18 | 458,584 rows (17.3%) | [109s]
  Batch   6/18 | 866,551 rows (32.7%) | [206s]
  Batch   9/18 | 1,330,514 rows (50.2%) | [314s]
  Batch  12/18 | 1,665,813 rows (62.9%) | [398s]
  Batch  15/18 | 2,178,088 rows (82.2%) | [516s]
  Batch  18/18 | 2,648,987 rows (100.0%) | [625s]

SAVED: submission_live_mard.parquet
  Rows: 2,648,987  NaN: 0  Keys: OK
  pred_30: [39.0, 404.0]
  pred_60: [39.0, 403.1]
  pred_90: [39.0, 404.1]
  pred_120: [39.0, 403.7]

Download submission_live_mard.parquet from the Output panel.


,id,source_file,date,pred_30,pred_60,pred_90,pred_120
0,16,AZT1D,2024-02-02 21:55:00,156.920595,150.511860,134.828996,127.794525
1,16,AZT1D,2024-02-02 22:10:00,156.803561,147.002276,130.915386,120.153001
2,16,AZT1D,2024-02-02 22:25:00,150.201334,134.731196,127.773378,119.005794
3,16,AZT1D,2024-02-02 22:40:00,146.729700,130.788767,120.079596,114.683920
4,16,AZT1D,2024-02-02 22:55:00,134.751678,127.774798,118.778706,116.498903


In [7]:
if LIVE_TARGETS_PATH.exists():
    tgts = pd.read_parquet(LIVE_TARGETS_PATH)
    tgts['date'] = pd.to_datetime(tgts['date'])
    scored = submission.merge(
        tgts[['id','date','target_30','target_60','target_90','target_120']],
        on=['id','date'], how='left'
    )
    print('\n' + '='*58)
    print('  LOCAL EVALUATION (targets.parquet)')
    print('='*58)
    for h in HORIZONS:
        p = scored[f'pred_{h}'].values
        t = scored[f'target_{h}'].values
        print(f'  {h:3d}-min | MARD={mard(p,t):6.2f}% | RMSE={rmse(p,t):6.2f} mg/dL')
    print('='*58)
    print('  Target (Rank 1) | Avg MARD~10.7% | DTS A-Zone~85.4%')
    print('='*58)
else:
    print('targets.parquet not found — submit to see leaderboard score.')

print('\nOutput files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<45s} {f.stat().st_size/(1024**2):.1f} MB')



  LOCAL EVALUATION (targets.parquet)
   30-min | MARD=  1.62% | RMSE=  3.68 mg/dL
   60-min | MARD=  1.65% | RMSE=  3.61 mg/dL
   90-min | MARD=  1.66% | RMSE=  3.62 mg/dL
  120-min | MARD=  1.67% | RMSE=  3.65 mg/dL
  Target (Rank 1) | Avg MARD~10.7% | DTS A-Zone~85.4%

Output files:
  __notebook__.ipynb                            0.1 MB
  feat_h_cols.json                              0.0 MB
  lgbm_h120.lgb                                 11.6 MB
  lgbm_h30.lgb                                  11.6 MB
  lgbm_h60.lgb                                  11.5 MB
  lgbm_h90.lgb                                  11.6 MB
  submission_live_mard.parquet                  103.2 MB
